# SQL assessment data to Silver

Reads the Azure SQL mirrored database from OneLake, standardizes the source tables, and writes managed Delta tables to `SilverLakehouse`. Change only the logical Fabric item name below when your lab uses a different name.

In [ ]:
%%configure -f
{
  "defaultLakehouse": { "name": "SilverLakehouse" }
}

In [ ]:
sql_mirror_item = "Property Assessment SQL Mirror"
source_schema = "dbo"

In [ ]:
import re
import requests
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

import notebookutils
workspace_id = notebookutils.runtime.context["currentWorkspaceId"]

def resolve_item_id(display_name, item_type):
    token = notebookutils.credentials.getToken("pbi")
    url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items?type={item_type}"
    response = requests.get(url, headers={"Authorization": f"Bearer {token}"}, timeout=60)
    response.raise_for_status()
    matches = [item for item in response.json()["value"] if item["displayName"] == display_name]
    if len(matches) != 1:
        raise ValueError(f"Expected one {item_type} named '{display_name}', found {len(matches)}")
    return matches[0]["id"]

def snake_case(name):
    name = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", name)
    return name.replace(" ", "_").lower()

mirror_id = resolve_item_id(sql_mirror_item, "MirroredDatabase")
spark.sql("CREATE SCHEMA IF NOT EXISTS dbo")
print(f"Resolved SQL mirror: {mirror_id}")

In [ ]:
table_map = {
    "Jurisdiction": ("dim_jurisdiction", ["JurisdictionId"]),
    "Neighborhood": ("dim_neighborhood", ["NeighborhoodId"]),
    "PropertyClass": ("dim_property_class", ["PropertyClassCode"]),
    "Parcel": ("dim_parcel", ["ParcelId"]),
    "Building": ("dim_building", ["BuildingId"]),
    "Assessment": ("fact_assessment", ["ParcelId", "TaxYear"]),
    "Sale": ("fact_sale", ["SaleId"]),
    "TaxRate": ("fact_tax_rate", ["TaxYear", "PropertyClassCode"]),
}

for source_table, (target_table, source_keys) in table_map.items():
    path = (
        f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/"
        f"{mirror_id}/Tables/{source_schema}/{source_table}"
    )
    frame = spark.read.format("delta").load(path)
    for column in frame.columns:
        frame = frame.withColumnRenamed(column, snake_case(column))
    for field in frame.schema.fields:
        if isinstance(field.dataType, StringType):
            frame = frame.withColumn(field.name, F.trim(F.col(field.name)))
    if target_table == "fact_assessment":
        frame = frame.withColumn("assessed_value", F.col("land_value") + F.col("improvement_value"))
    keys = [snake_case(key) for key in source_keys]
    frame = frame.dropna(subset=keys).dropDuplicates(keys)
    (frame.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"dbo.{target_table}"))
    print(f"{target_table}: {frame.count()} rows")

In [ ]:
assert spark.table("dbo.dim_parcel").select("parcel_id").distinct().count() == spark.table("dbo.dim_parcel").count()
assert spark.table("dbo.fact_assessment").filter(F.col("assessed_value") <= 0).count() == 0
assert spark.table("dbo.dim_jurisdiction").filter(F.col("is_synthetic") != True).count() == 0
display(spark.table("dbo.fact_assessment").groupBy("tax_year").agg(F.count("*").alias("rows"), F.avg("assessed_value").alias("average_assessed_value")))